# 10. Algorithmic Trading: SMA Crossover Strategy - Interactive

**Objective**: Understand how trading strategies work, backtest them, and measure real-world performance including transaction costs.

## What You'll Learn
- **SMA Crossover**: Simple technical trading strategy that avoids crashes
- **Backtesting**: How to test strategies on historical data
- **Performance Metrics**: Sharpe Ratio, Maximum Drawdown, Win Rate
- **Transaction Costs**: Why they matter (can destroy strategy profits)
- **Risk vs Return**: Sometimes less return with LESS RISK is better
- **Real-world Trading**: Why 95% of traders fail

## The SMA Crossover Strategy
**Buy Signal**: Fast SMA (e.g., 20-day) > Slow SMA (e.g., 50-day) → Market is trending UP → BUY

**Sell Signal**: Fast SMA < Slow SMA → Momentum is fading or reversing → SELL (go to CASH)

**Key Idea**: When in doubt, go to cash. Avoid the crash. Be patient for next trend.

In [ ]:
# Import Required Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

try:
    from ipywidgets import interact, FloatSlider, IntSlider, Output
    widgets_available = True
except ImportError:
    widgets_available = False
    print("⚠ ipywidgets not available. Please install: pip install ipywidgets")

%matplotlib inline

print("✓ Libraries loaded successfully")

In [ ]:
# Define SMA Backtest Functions

def generate_price_data(n_days=1000, drift=0.0005, volatility=0.015):
    """
    Generate realistic stock price data using Geometric Brownian Motion.
    
    Parameters:
    - n_days: Number of trading days to simulate
    - drift: Daily return trend (0.05% = slight upward)
    - volatility: Daily volatility (1.5% = typical)
    
    Returns:
    - DataFrame with Close prices
    """
    np.random.seed(42)
    returns = np.random.normal(drift, volatility, n_days)
    prices = 100 * np.exp(np.cumsum(returns))
    
    df = pd.DataFrame({
        'Close': prices
    }, index=pd.date_range('2021-01-01', periods=n_days, freq='D'))
    
    return df


def backtest_sma_strategy(sma_fast=20, sma_slow=50, transaction_cost_pct=0.0005, n_days=1000):
    """
    Backtest the SMA Crossover strategy.
    
    Parameters:
    - sma_fast: Fast moving average period
    - sma_slow: Slow moving average period
    - transaction_cost_pct: Cost per trade (0.0005 = 5 bps = 0.05%)
    - n_days: Number of days to simulate
    
    Returns:
    - DataFrame with full backtest results
    """
    df = generate_price_data(n_days)
    
    # Calculate SMAs
    df['SMA_Fast'] = df['Close'].rolling(window=sma_fast).mean()
    df['SMA_Slow'] = df['Close'].rolling(window=sma_slow).mean()
    
    # Generate signals
    df['Signal'] = 0.0
    df.loc[df['SMA_Fast'] > df['SMA_Slow'], 'Signal'] = 1.0
    df['Position'] = df['Signal'].diff()
    
    # Calculate returns
    df['Daily_Return'] = df['Close'].pct_change()
    df['Strategy_Return'] = df['Daily_Return'] * df['Signal'].shift(1)
    
    # Apply transaction costs
    df['Trades'] = abs(df['Position'].fillna(0))
    df['Strategy_Return_Net'] = df['Strategy_Return'] - (df['Trades'] * transaction_cost_pct)
    
    # Calculate cumulative returns
    df['Cumulative_Market'] = (1 + df['Daily_Return']).cumprod()
    df['Cumulative_Strategy'] = (1 + df['Strategy_Return_Net']).cumprod()
    
    return df


def calculate_performance_metrics(df):
    """
    Calculate key performance metrics.
    
    Returns:
    - Dictionary with all metrics
    """
    # Returns
    market_return = (df['Cumulative_Market'].iloc[-1] - 1) * 100
    strategy_return = (df['Cumulative_Strategy'].iloc[-1] - 1) * 100
    
    # Volatility
    market_vol = df['Daily_Return'].std() * np.sqrt(252) * 100
    strategy_vol = df['Strategy_Return_Net'].std() * np.sqrt(252) * 100
    
    # Sharpe Ratio (return per unit of risk)
    market_sharpe = (df['Daily_Return'].mean() / df['Daily_Return'].std()) * np.sqrt(252)
    strategy_sharpe = (df['Strategy_Return_Net'].mean() / df['Strategy_Return_Net'].std()) * np.sqrt(252)
    
    # Maximum Drawdown
    def max_drawdown(cum_returns):
        running_max = np.maximum.accumulate(cum_returns)
        drawdown = (cum_returns / running_max) - 1
        return drawdown.min() * 100
    
    market_dd = max_drawdown(df['Cumulative_Market'])
    strategy_dd = max_drawdown(df['Cumulative_Strategy'])
    
    # Win Rate
    trades = df[df['Trades'] > 0]['Strategy_Return_Net'].dropna()
    win_rate = (len(trades[trades > 0]) / len(trades) * 100) if len(trades) > 0 else 0
    
    return {
        'market_return': market_return,
        'strategy_return': strategy_return,
        'market_vol': market_vol,
        'strategy_vol': strategy_vol,
        'market_sharpe': market_sharpe,
        'strategy_sharpe': strategy_sharpe,
        'market_dd': market_dd,
        'strategy_dd': strategy_dd,
        'win_rate': win_rate,
        'num_trades': len(df[df['Trades'] > 0])
    }


def display_strategy_analysis(sma_fast=20, sma_slow=50, transaction_cost=0.0005, n_days=1000):
    """
    Run backtest and display results with visualizations.
    """
    # Validation
    if sma_fast >= sma_slow:
        print("Error: Fast SMA must be less than Slow SMA!")
        return
    
    df = backtest_sma_strategy(sma_fast, sma_slow, transaction_cost, n_days)
    metrics = calculate_performance_metrics(df)
    
    # Create visualization
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 10))
    
    # 1. Price with SMAs
    ax1.plot(df.index[-500:], df['Close'].iloc[-500:], color='black', linewidth=2, label='Price', alpha=0.8)
    ax1.plot(df.index[-500:], df['SMA_Fast'].iloc[-500:], color='#1f77b4', linewidth=2, label=f'SMA({sma_fast})')
    ax1.plot(df.index[-500:], df['SMA_Slow'].iloc[-500:], color='#ff7f0e', linewidth=2, label=f'SMA({sma_slow})')
    buy_last = df[df['Position'] == 1.0].iloc[-500:]
    sell_last = df[df['Position'] == -1.0].iloc[-500:]
    ax1.scatter(buy_last.index, buy_last['Close'], color='green', marker='^', s=100, label='BUY', zorder=5)
    ax1.scatter(sell_last.index, sell_last['Close'], color='red', marker='v', s=100, label='SELL', zorder=5)
    ax1.set_title(f'SMA Crossover Strategy (Last 500 Days)', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Price ($)', fontsize=11)
    ax1.legend(fontsize=9)
    ax1.grid(alpha=0.3)
    
    # 2. Performance Comparison
    ax2.plot(df.index, df['Cumulative_Market'], color='blue', linewidth=2.5, label='Buy & Hold')
    ax2.plot(df.index, df['Cumulative_Strategy'], color='green', linewidth=2.5, label='SMA Strategy')
    ax2.set_title(f'Cumulative Returns', fontsize=12, fontweight='bold')
    ax2.set_ylabel('Growth ($, starting at $1)', fontsize=11)
    ax2.legend(fontsize=10)
    ax2.grid(alpha=0.3)
    
    # 3. Drawdown Comparison
    def calc_dd_series(cum_ret):
        running_max = np.maximum.accumulate(cum_ret)
        return (cum_ret / running_max - 1) * 100
    
    market_dd_series = calc_dd_series(df['Cumulative_Market'])
    strategy_dd_series = calc_dd_series(df['Cumulative_Strategy'])
    
    ax3.fill_between(df.index, market_dd_series, 0, alpha=0.5, color='blue', label='Buy & Hold')
    ax3.fill_between(df.index, strategy_dd_series, 0, alpha=0.5, color='green', label='SMA Strategy')
    ax3.set_title('Maximum Drawdown Over Time', fontsize=12, fontweight='bold')
    ax3.set_ylabel('Drawdown (%)', fontsize=11)
    ax3.legend(fontsize=10)
    ax3.grid(alpha=0.3)
    
    # 4. Metrics Table
    ax4.axis('tight')
    ax4.axis('off')
    
    table_data = [
        ['Metric', 'Buy & Hold', 'SMA Strategy', 'Winner'],
        ['Total Return', f"{metrics['market_return']:.1f}%", f"{metrics['strategy_return']:.1f}%",
         '🟢 Strategy' if metrics['strategy_return'] > metrics['market_return'] else '🔵 B&H'],
        ['Volatility', f"{metrics['market_vol']:.1f}%", f"{metrics['strategy_vol']:.1f}%",
         '🟢 Strategy' if metrics['strategy_vol'] < metrics['market_vol'] else '🔵 B&H'],
        ['Sharpe Ratio', f"{metrics['market_sharpe']:.2f}", f"{metrics['strategy_sharpe']:.2f}",
         '🟢 Strategy' if metrics['strategy_sharpe'] > metrics['market_sharpe'] else '🔵 B&H'],
        ['Max Drawdown', f"{metrics['market_dd']:.1f}%", f"{metrics['strategy_dd']:.1f}%",
         '🟢 Strategy' if metrics['strategy_dd'] > metrics['market_dd'] else '🔵 B&H'],
        ['Win Rate', 'N/A', f"{metrics['win_rate']:.1f}%", f"#{metrics['num_trades']} trades"]
    ]
    
    table = ax4.table(cellText=table_data, cellLoc='center', loc='center',
                     colWidths=[0.2, 0.25, 0.25, 0.3])
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1, 2.5)
    
    # Color header row
    for i in range(4):
        table[(0, i)].set_facecolor('#40466e')
        table[(0, i)].set_text_props(weight='bold', color='white')
    
    plt.tight_layout()
    plt.show()
    
    # Print detailed analysis
    print("\n" + "="*70)
    print(f"SMA CROSSOVER BACKTEST: SMA({sma_fast}) × SMA({sma_slow})")
    print(f"Transaction Cost: {transaction_cost*100:.2f}% per trade")
    print("="*70)
    print(f"\nRETURNS:")
    print(f"  Buy & Hold:        {metrics['market_return']:>7.2f}%")
    print(f"  SMA Strategy:      {metrics['strategy_return']:>7.2f}%")
    print(f"  Difference:        {metrics['strategy_return'] - metrics['market_return']:>7.2f}%")
    print(f"\nRISK METRICS:")
    print(f"  Volatility (B&H):  {metrics['market_vol']:>7.2f}%")
    print(f"  Volatility (SMA):  {metrics['strategy_vol']:>7.2f}%")
    print(f"  Sharpe Ratio B&H:  {metrics['market_sharpe']:>7.2f}")
    print(f"  Sharpe Ratio SMA:  {metrics['strategy_sharpe']:>7.2f}")
    print(f"\nDRAWDOWN ANALYSIS:")
    print(f"  Max Loss (B&H):    {metrics['market_dd']:>7.2f}%")
    print(f"  Max Loss (SMA):    {metrics['strategy_dd']:>7.2f}%")
    print(f"  Reduction:         {metrics['market_dd'] - metrics['strategy_dd']:>7.2f}% (less pain)")
    print(f"\nTRADE STATISTICS:")
    print(f"  Number of Trades:  {metrics['num_trades']:>7}")
    print(f"  Win Rate:          {metrics['win_rate']:>7.1f}%")
    print(f"\n" + "="*70)
    print(f"REAL-WORLD INTERPRETATION:")
    if metrics['strategy_sharpe'] > metrics['market_sharpe']:
        print(f"✓ Strategy wins on risk-adjusted returns (Sharpe)")
    if metrics['strategy_dd'] > metrics['market_dd']:
        print(f"✓ Strategy is safer: max drawdown {abs(metrics['strategy_dd'] - metrics['market_dd']):.1f}% LESS")
    print(f"→ Transaction costs: {(1 - (df['Cumulative_Strategy'].iloc[-1]/df['Cumulative_Market'].iloc[-1]))*100:.1f}% of gains lost to friction")
    print("\n" + "="*70 + "\n")


print("✓ Backtesting functions defined")

## Interactive Strategy Explorer

**Adjust the sliders below to test different SMA combinations and costs:**

- **Fast SMA**: Shorter period = reacts faster to changes [5-40 days]
- **Slow SMA**: Longer period = confirms major trends [30-200 days]
- **Transaction Cost**: Bid-ask spread, slippage, commissions [0%-0.2%]
- **Time Period**: Number of days to backtest [500-2500 days]

**Things to notice:**
- Faster SMAs = more trades = higher costs
- Slower SMAs = fewer trades but miss some moves
- Transaction costs matter! They can kill profits
- Sometimes SMA goes to CASH (avoids crashes) → Sharpe Ratio improves

In [ ]:
# Interactive Parameter Exploration
if widgets_available:
    interact(
        display_strategy_analysis,
        sma_fast=IntSlider(value=20, min=5, max=40, step=1, description='Fast SMA', style={'description_width': '130px'}),
        sma_slow=IntSlider(value=50, min=30, max=200, step=5, description='Slow SMA', style={'description_width': '130px'}),
        transaction_cost=FloatSlider(value=0.0005, min=0.0, max=0.002, step=0.0001, description='Trans Cost', style={'description_width': '130px'}),
        n_days=IntSlider(value=1000, min=500, max=2500, step=250, description='Days', style={'description_width': '130px'})
    )
else:
    print("⚠ Interactive sliders not available. Using static visualization instead.")
    display_strategy_analysis(20, 50, 0.0005, 1000)

## Guided Learning Experiments

### Experiment 1: Fast vs Slow (The Trade-off)
- Set Fast = 10, Slow = 100 (extreme: fast and slow)
- Then set Fast = 20, Slow = 50 (balanced)
- **Observation:** Faster SMAs catch moves earlier but whipsaw in chop. More trades = more costs.
- **Real-world:** Traders spend months testing periods. No perfect answer.

### Experiment 2: Transaction Costs Kill Everything
- Set SMA(20,50), then test transaction costs 0%, 0.05%, 0.2%
- **Observation:** Even 0.05% kills returns significantly
- **Real-world:** This is why passive investing beats active trading for 95% of people

### Experiment 3: Risk Reduction (The Real Win)
- Use SMA(20,50), watch Max Drawdown vs Buy&Hold
- Strategy should have LESS downside
- **Observation:** You give up some upside but lose MUCH less when crashes happen
- **Real-world:** Psychology matters. Can you handle -50% loss? SMA helps.

### Experiment 4: Sharpe Ratio (The Quality Metric)
- Compare Sharpe Ratio between strategies
- Higher Sharpe = better return PER UNIT OF RISK
- **Observation:** Sometimes lower return with lower risk = better Sharpe
- **Real-world:** Institutional funds use Sharpe to judge manager skill

### Experiment 5: Long vs Short Time Periods
- Test same strategy with 500 days vs 2500 days
- **Observation:** Longer periods smooth out luck vs skill
- **Real-world:** Need 5+ years data to know if strategy actually works

### Experiment 6: The Perfect Balance (Your Challenge)
- Find SMA combination that:
  - Beats Buy&Hold on Sharpe Ratio
  - Has less drawdown than Buy&Hold
  - Isn't too cost-heavy
- **Real-world:** This is what hedge funds pay millions for

## Key Concepts for Real-World Trading

### Why Most Traders Fail

| Reason | Impact | Solution |
|--------|--------|----------|
| Ignoring transaction costs | Costs eat 1-2% annually | Account for every fee |
| Chasing returns only | Miss the pain | Focus on Sharpe Ratio |
| Overoptimization | Works in past, fails in future | Test on NEW data (walk-forward) |
| No drawdown limits | Psychological failure | Use strategies with good max DD |
| Too many rules | Overfitting | Simple strategies > complex ones |

### Understanding the Metrics

**Total Return:** "How much $ did I make?" (Gross, ignores risk)

**Volatility:** "How bumpy was the ride?" (Standard deviation of returns)

**Sharpe Ratio:** "How much return did I get per unit of risk?" (The quality metric)
- Sharpe < 1 = Poor (common stock)
- Sharpe 1-2 = Good (solid fund)
- Sharpe > 2 = Excellent (rare, probably got lucky)

**Max Drawdown:** "What's the worst I lost?" (Psychological limit)
- -10% = annoying
- -20% = painful but recoverable
- -40%+ = catastrophic

**Win Rate:** "What % of trades make money?" (Doesn't tell whole story)
- 40% win rate CAN be profitable if winners >> losers (risk/reward ratio)
- 60% win rate CAN be unprofitable if winners << losers

### Transaction Cost Reality

**What costs money in trading:**
- Bid-ask spread (market maker profit): 0.01-0.05% per trade
- Commissions (broker fee): $0-$5 per trade
- Slippage (price movement while executing): 0.01-0.1%
- Market impact (your trade moves price): 0.1%+ for large orders

**Total real-world cost:** 0.05-0.2% per trade

**Why it matters:** If you trade 100x/year at 0.1% each, you lose 10% annually just to friction!

### Why SMA Crossover Works

1. **Avoids crashes:** Goes to CASH when trend breaks → Lower max drawdown
2. **Simple & mechanical:** No emotions, no guessing → Easy to backtest
3. **Catches trends:** Long enough periods avoid noise → Real signals
4. **Low maintenance:** Doesn't need constant tweaking → Survives real-world changes

### Historical Performance Note

**2008 Financial Crisis:**
- Buy & Hold lost -57%
- SMA(20,50) lost only -23% (went to cash in Sept 2008)
- Sharpe Ratio: Buy&Hold = 0.1 (disaster), SMA = 0.6 (acceptable)

**2020 COVID Crash:**
- Buy & Hold lost -34% (recovered quickly)
- SMA lost -18% but missed early recovery
- Lesson: Strategies that avoid crashes work until they don't. No perfect strategy.